In [48]:
from plutonlib.obs_properties import ObsProperties
import astropy.units as u
from plutonlib.utils import py_reload_all
py_reload_all()

plutonlib.__init__ Last Saved: Mon Oct 26 11:52:32 2026
plutonlib._version Last Saved: Mon Oct 26 11:52:32 2026
plutonlib.colours Last Saved: Thu Nov  6 13:25:50 2025
plutonlib.compression Last Saved: Thu May 14 14:21:42 2026
plutonlib.config Last Saved: Mon Nov  2 11:26:23 2026
plutonlib.fancy_plot Last Saved: Tue Nov 17 13:32:19 2026
plutonlib.pbs_job Last Saved: Mon Nov 16 11:58:23 2026
plutonlib.plot_data Last Saved: Tue Nov 17 13:21:15 2026
plutonlib.simulation_info Last Saved: Thu Oct 29 14:49:57 2026
plutonlib.surface_brightness Last Saved: Tue Nov 24 11:59:20 2026
plutonlib.utils Last Saved: Thu Oct 29 15:34:57 2026
plutonlib.simulation_report Last Saved: Mon Oct 12 14:02:29 2026
plutonlib.units Last Saved: Mon Oct 26 15:05:30 2026
plutonlib.splines Last Saved: Wed Nov 18 14:49:28 2026
plutonlib.analysis Last Saved: Mon Nov  9 13:51:53 2026
plutonlib.morphology Last Saved: Wed Nov 18 15:22:55 2026
plutonlib.peaks Last Saved: Mon Nov 16 16:46:26 2026
plutonlib.read_write Last Sa

In [ ]:
from pathlib import Path
from collections import defaultdict

IMAGE_DIR = Path("~/jet_gui/images").expanduser()


def get_all_sources():
    source_dirs = [x for x in IMAGE_DIR.iterdir() if x.is_dir()]
    source_dict = {f.name: f for f in source_dirs}
    return source_dict

def get_source_images(source_id):
    """Gets each image layer PNG associated with a source id"""
    source_dir = IMAGE_DIR / source_id
    if not source_dir.is_dir():
        raise FileNotFoundError(f"source image directory: {source_dir} not found!")
    image_layers = ["lr", "lr-info", "hr", "hr-splines"]
    image_paths = {}
    for layer in image_layers:
        im_path = source_dir / f"{layer}.png"
        if im_path.is_file():
            image_paths[layer] = im_path
        else:
            print(f"Missing {layer} image for source: {source_id}")
            image_paths[layer] = None
    return image_paths

In [39]:
get_source_images("source_1")

{'lr': PosixPath('/home/laani/jet_gui/images/source_1/lr.png'),
 'lr-info': PosixPath('/home/laani/jet_gui/images/source_1/lr-info.png'),
 'hr': PosixPath('/home/laani/jet_gui/images/source_1/hr.png'),
 'hr-splines': PosixPath('/home/laani/jet_gui/images/source_1/hr-splines.png')}

In [ ]:
%gui qt

from PySide6.QtWidgets import QApplication, QWidget
import sys

app = QApplication.instance() or QApplication([])
window = QWidget()
window.show()

# Start the event loop.
# app.exec()

qt.qpa.wayland.textinput: virtual void QtWaylandClient::QWaylandTextInputv3::disableSurface(wl_surface*) Trying to disable 0x55ae3138d080 but 0x0 is focused
qt.qpa.wayland.textinput: virtual void QtWaylandClient::QWaylandTextInputv3::disableSurface(wl_surface*) Trying to disable 0x55ae3138d080 but 0x0 is focused


## Telescope params

In [57]:
obs = ObsProperties(freqs=[0.15],redshifts=[0.1])
obs.pixel_size_kpc


<Quantity 3.80908843 kpc>

In [58]:
from astropy import cosmology as cosmo  # Astropy cosmology

target_resolution = 6*u.arcsec #target telescope res in arcsec 
redshifts = [0.03,0.05,0.1]
beam_pixel_ratio = 3 #beam ~3x pixel size



In [60]:
import plutonlib.utils as pu
import plutonlib.read_write as prw
from plutonlib.obs_properties import create_obs_grid
# import plutonlib.simulations as ps

import plutokore.radio as pk_radio

import numpy as np

from astropy import units as u
from astropy import cosmology as cosmo  # Astropy cosmology


def calc_surface_brightness_praise(sdata,obs_props,part_outputs="last"):
    """Calculates the particle emssion using PRAiSE (plutokore: pk_radio) under adiabatic, 
    sychrotron and inverse compton losses. Surface brightness is then calculated by integrating 
    emissivity with raytracing.    

    Args:
        sdata (SimulationData): SimulationData object, used for grid and wdir info
        obs_props (ObsProperties): ObsProperties object, used to store observing properties e.g. redshift, angle, freqs
        part_outputs (str, optional): _description_. Defaults to "last".

    Returns:
        sb_arr: surface brightness array
    """

    obs_grid = create_obs_grid(sdata,obs_props)
    pk_sim = sdata.to_plutokore() #convert SimulationData object to plutokore PlutoSimulation
    part_outputs = [prw.get_particle_outputs(sdata.wdir)] if part_outputs == "last" else part_outputs
    particle_spacing = sdata.part_to_simtime(part_outputs[0]) / part_outputs[0]

    #load all the available particle files from hdf5
    particle_data = sdata.load_particle_data(part_output=part_outputs[-1],force_check=False) #NOTE force_check turned off here to allow faster calcs with no checks
    alias_map = {"rho": "density", "prs": "pressure", "tr1": "tracer"} #NOTE use to convert to praise format
    for short_key, long_key in alias_map.items():
        if long_key not in particle_data and short_key in particle_data:
            particle_data[long_key] = particle_data[short_key]

    particle_times = particle_data["particle_times"]
    particle_emis = pk_radio.praise2.praise(
        sim=pk_sim,
        max_output=part_outputs[-1],
        emit_outputs=part_outputs, #calc emission for these outputs
        output_system="particles", #idx in grid or particles
        freqs=obs_props.freqs.si.value, #list of GHz freqs to calc for 
        part_data=particle_data, 
        part_times=particle_times,
        particle_spacing=particle_spacing * u.Myr, #particle outputs per Myr 
        redshift=obs_props.redshifts, #list of redshifts
        lst_index=2, #last shock time index, lowest to highest str -> 2 = only strong shocks
        losses=4, #idx to include losses: adiab, synch, inverse compton losses -> 4 = include all losses
    )

    # Remove the NaNs for each output 
    sb_arr = []
    for i in  range(0, len(part_outputs), 1):    
        part_ind = part_outputs[i]
        nan_mask = ~np.isnan(particle_data["id"][:, part_ind]) 
        emis = particle_emis['full'][i]['emis'][nan_mask]        # (n_p, n_f, n_z)
        n_p, n_f, n_z = emis.shape
        emis_flat = emis.reshape(n_p, n_f * n_z) #NOTE since raytracing is 2D, make second column include both all freqs and all redshifts


        part_coords = np.c_[
            (
                particle_data["x1"][:, part_ind][nan_mask],
                particle_data["x2"][:, part_ind][nan_mask],
                particle_data["x3"][:, part_ind][nan_mask],
            )
        ]
        
        # set up particle velocities
        vel_vec = np.c_[
            particle_data["vx1"][:, part_ind][nan_mask],
            particle_data["vx2"][:, part_ind][nan_mask],
            particle_data["vx3"][:, part_ind][nan_mask],
        ]

        integrated_emissivity = pk_radio.raytracing.raytrace_particles_multiple_freq(
            grid=(obs_grid["grid_mx"], obs_grid["grid_my"]),
            ray_depth_lim=(obs_props.ray_depth_min, obs_props.ray_depth_max),
            rot_mat=obs_props.rot_matrix,
            s= obs_props.alpha_inj,
            delta_r=obs_props.delta_r,
            coords=part_coords,
            dist_upper_bound = 2,      # <-- NOTE the praise default is 20... this will mess up your results. 2 is better
            vel_vec=vel_vec,
            obs_normal=[0, 1, 0],
            particle_emissivities=emis_flat #only the non-nan particles, all frequencies, and the ith snapshot 
        )

        #NOTE particle_emis has shape (n_part,n_freq,n_z)
        nx, ny = integrated_emissivity.shape[:2]
        integrated_emissivity = integrated_emissivity.reshape(nx, ny, n_f, n_z)

        # we multiple our integrated emissivity by kpc (to account for integration), and divide by 4pi to account for solid angle
        surface_brightness = (integrated_emissivity * u.kpc / (4 * np.pi)).to(u.mJy / u.beam, equivalencies=u.beam_angular_area(obs_props.omega_beam))
        surface_brightness = np.nan_to_num(surface_brightness, nan=0.0, posinf=0.0, neginf=0.0)
        sb_arr.append(surface_brightness)

    return sb_arr
